# Generation and evaluation (debug)

Goal:
- reuse the retrieval pipeline built earlier
- retrieve top-k chunks for each question
- build a prompt from the retrieved context
- generate an answer with Llama 3.2 1B
- evaluate predictions with F1, EM, Recall@k, and latency

Important:
- this notebook is for debugging and validating the full pipeline on a small subset first (Testing on CPU directly from the notebook)
- the final large scale experiment will later be moved to a Python script and run with Slurm on GPU (To have access to GPUs with LMU servers are only via slurm)
- retrieval settings to evaluate later:
  - chunk_size in {32, 128, 256}
  - top_k in {1, 5, 10}

In [1]:
# Imports for data loading, indexing, generation, and evaluation

import json
import time
import re
import string
import numpy as np
import pandas as pd
import faiss

from pathlib import Path
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

/home/a/arfaoui/miniconda3/envs/AAML/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

data_dir = Path("/home/a/arfaoui/rag_project/data")

# Retrieval settings that will later be varied systematically
chunk_sizes = [32, 128, 256]
top_k_values = [1, 5, 10]

# For testing, we start with one configuration
debug_chunk_size = 128
debug_top_k = 5

# Start with a small number of questions for fast testing
debug_n_questions = 5
# print to check
print("Data directory:", data_dir)
print("Test chunk size:", debug_chunk_size)
print("Test top-k:", debug_top_k)
print("Test number of questions:", debug_n_questions)

Data directory: /home/a/arfaoui/rag_project/data
Test chunk size: 128
Test top-k: 5
Test number of questions: 5


In [3]:
# Load saved embeddings and metadata for all chunk sizes
# We need these to rebuild or reuse the retrieval pipeline inside this notebook 

all_embeddings = {}
all_metadata = {}

for size in chunk_sizes:
    # Load embedding matrix
    emb_path = data_dir / f"embeddings_{size}.npy"
    embeddings = np.load(emb_path)
    all_embeddings[size] = embeddings

    # Load chunk metadata
    meta_path = data_dir / f"chunks_metadata_{size}.json"
    with open(meta_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    all_metadata[size] = metadata

    print(f"\nchunk_size={size}")
    print("Embeddings shape:", embeddings.shape)
    print("Metadata entries:", len(metadata))


chunk_size=32
Embeddings shape: (292199, 384)
Metadata entries: 292199

chunk_size=128
Embeddings shape: (96686, 384)
Metadata entries: 96686

chunk_size=256
Embeddings shape: (69845, 384)
Metadata entries: 69845


In [4]:
# Load precomputed FAISS indexes from disk

faiss_indexes = {}

for size in chunk_sizes:
    index_path = data_dir / f"faiss_index_{size}.index"
    index = faiss.read_index(str(index_path))
    
    faiss_indexes[size] = index
    print(f"Loaded FAISS index for chunk_size={size}")
    print("Index size:", index.ntotal)

Loaded FAISS index for chunk_size=32
Index size: 292199
Loaded FAISS index for chunk_size=128
Index size: 96686
Loaded FAISS index for chunk_size=256
Index size: 69845


In [6]:
# Load the fixed HotpotQA sample from disk
# The sample file is in JSON Lines format, so we use the datasets library.

sample_path = data_dir / "hotpotqa_full_sample.json"
hotpot_sample = load_dataset("json", data_files=str(sample_path))["train"]
# print to check 
print("Loaded questions:", len(hotpot_sample))
print("Example question:", hotpot_sample[0]["question"])
print("Example answer:", hotpot_sample[0]["answer"])
# Create a small testing subset of questions
# We use only the first N questions for quick pipeline testing.

debug_sample = hotpot_sample.select(range(debug_n_questions))

print("Debug subset size:", len(debug_sample))
print("First debug question:", debug_sample[0]["question"])

Loaded questions: 7405
Example question: Were Scott Derrickson and Ed Wood of the same nationality?
Example answer: yes
Debug subset size: 5
First debug question: Were Scott Derrickson and Ed Wood of the same nationality?


In [7]:
# Load the same embedding model used earlier for chunk embeddings
# We need it here to embed questions for retrieval.

embedding_model_name = "BAAI/bge-small-en-v1.5"
embed_model = SentenceTransformer(embedding_model_name)

print("Embedding model loaded:", embedding_model_name)

Embedding model loaded: BAAI/bge-small-en-v1.5


In [8]:
# Same function from notebook 05 to retrieve top-k chunks for a given question

def retrieve_top_k(question, index, metadata, k=5):
    """
    Given a question, retrieve top-k most similar chunks.
    
    Returns:
    - retrieved_chunks: list of metadata entries
    - scores: similarity scores
    """
    
    # BGE models expect "query: " prefix for queries
    query_text = "query: " + question
    
    # Encode query
    query_embedding = embed_model.encode([query_text])
    
    # Convert to float32 (FAISS requirement)
    query_embedding = np.array(query_embedding).astype("float32")
    
    # Search in FAISS index
    D, I = index.search(query_embedding, k)
    
    # Retrieve corresponding chunks
    retrieved_chunks = [metadata[idx] for idx in I[0]]
    scores = D[0]
    
    return retrieved_chunks, scores

In [9]:
# Build a single text context string from retrieved chunks
# This will later be passed into the LLM prompt, without it the retrieved chunks are useful information but can't give it to the model
# This makes it:
#  -human-readable
#  -structured for the LLM
#  -easy to debug


def build_retrieved_context(retrieved_chunks):
    """
    Concatenate retrieved chunks into one context string.
    """
    parts = []

    for i, chunk in enumerate(retrieved_chunks, start=1):
        parts.append(f"[Chunk {i}] Title: {chunk['title']}\n{chunk['text']}")

    return "\n\n".join(parts)

## LLM generation setup

In this section, we load the fixed generation model and define a prompt template.

Important:
- the LLM is fixed across all experiments
- the prompt must remain constant across all configurations
- the model should answer using only the retrieved context
- this stage is still for testing on a small subset before the final Slurm run  
  
## Prompt design

We use one fixed prompt template across all experiments.

Design goals:
- force the model to rely only on retrieved context
- keep the prompt simple and constant

In [10]:
# Load the generation model and tokenizer
# We keep the model fixed across all experiments so that we isolate the effect of the chunk size and top-k 

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

llm_name = "meta-llama/Llama-3.2-1B-Instruct"

# Load tokenizer
llm_tokenizer = AutoTokenizer.from_pretrained(llm_name)

# Load causal language model
# device_map="auto" lets transformers place the model on available hardware
# torch_dtype="auto" uses a suitable dtype automatically when possible
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype="auto",
    device_map="auto"
)
# print to check 
print("LLM tokenizer loaded:", llm_name)
print("LLM model loaded:", llm_name)

/home/a/arfaoui/miniconda3/envs/AAML/lib/python3.10/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


LLM tokenizer loaded: meta-llama/Llama-3.2-1B-Instruct
LLM model loaded: meta-llama/Llama-3.2-1B-Instruct


In [11]:
# Build a fixed prompt from retrieved context and question
# We want the model to extract a short answer from the context.

def build_prompt(question, context):
    prompt = f"""Context:
{context}

Question: {question}

Short answer (just the key phrase, no full sentences):"""
    return prompt
    

In [12]:
# Test retrieval + context building + prompt construction on one example question

example = debug_sample[0]

question = example["question"]
correct_answer = example["answer"]

# Use the balanced debug setting first
index = faiss_indexes[debug_chunk_size]
metadata = all_metadata[debug_chunk_size]

retrieved_chunks, retrieval_scores = retrieve_top_k(
    question=question,
    index=index,
    metadata=metadata,
    k=debug_top_k
)

context = build_retrieved_context(retrieved_chunks)
prompt = build_prompt(question, context)

print("Question:", question)
print("correct ansewer:", correct_answer)
print("\n--- Prompt preview ---\n")
print(prompt[:2000])  # preview first part only

Question: Were Scott Derrickson and Ed Wood of the same nationality?
correct ansewer: yes

--- Prompt preview ---

Context:
[Chunk 1] Title: Scott Derrickson
Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Universe installment, "Doctor Strange."

[Chunk 2] Title: Ed Wood
Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer, and director.

[Chunk 3] Title: Ed Wood (film)
Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.  The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.  Sarah Jessica Par

In [13]:
# Generate an answer from the LLM given a prompt
# Decode only the newly generated continuation, not the entire prompt.

def generate_answer(prompt, tokenizer, model, max_new_tokens=32):
    """
    Generate a short answer from the LLM.
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Only decode the newly generated tokens
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # Keep only the first line
    answer = answer.split("\n")[0].strip()

    return answer

In [14]:
# Run one full pipeline example: retrieval → prompt → generation

start_time = time.time()

generated_answer = generate_answer(
    prompt,
    tokenizer=llm_tokenizer,
    model=llm_model
)

latency = time.time() - start_time

print("=== MODEL OUTPUT ===\n")
print(generated_answer)

print("\n=== LATENCY ===")
print(f"{latency:.4f} seconds")

/home/a/arfaoui/miniconda3/envs/AAML/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/a/arfaoui/miniconda3/envs/AAML/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


=== MODEL OUTPUT ===

No.

=== LATENCY ===
26.5410 seconds


In [15]:
import string

def normalize_text(text):
    """
    Normalize text for fair comparison.

    - lowercase
    - remove punctuation
    - strip whitespace
    """

    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.strip()

    return text

In [16]:
norm_pred = normalize_text(generated_answer )
norm_gt = normalize_text(correct_answer)

print("Normalized prediction:", norm_pred)
print("Normalized correct_answer:", norm_gt)

Normalized prediction: no
Normalized correct_answer: yes


In [17]:
# Evaluation functions

# Exact Match after normalization
# Returns 1 if prediction and correct answer match exactly, else 0.

def compute_em(prediction, correct_answer):
    pred_norm = normalize_text(prediction)
    gt_norm = normalize_text(correct_answer)
    return int(pred_norm == gt_norm)
    
# Token-level F1 score
# This measures overlap between predicted answer tokens and correct answer tokens.

def compute_f1(prediction, correct_answer):
    pred_tokens = normalize_text(prediction).split()
    gt_tokens = normalize_text(correct_answer).split()

    # Handle empty cases safely
    if len(pred_tokens) == 0 and len(gt_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return 0.0

    # Count token overlap
    common_tokens = set(pred_tokens) & set(gt_tokens)
    num_same = sum(min(pred_tokens.count(tok), gt_tokens.count(tok)) for tok in common_tokens)

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    f1 = 2 * precision * recall / (precision + recall)

    return f1
    
# Recall@k for retrieval
# Returns 1 if the answer appears in at least one retrieved chunk, else 0.

def compute_recall_at_k(retrieved_chunks, correct_answer):
    gt_norm = normalize_text(correct_answer)

    for chunk in retrieved_chunks:
        chunk_text_norm = normalize_text(chunk["text"])
        if gt_norm in chunk_text_norm:
            return 1

    return 0
# Recall@k_titles 
# Unlike the simple answer-based Recall@k (which checks if the answer string
# appears in retrieved chunks), this metric evaluates whether the retriever
# successfully retrieved the correct supporting documents.
#
# In HotpotQA, each question is associated with one or more supporting titles
# (documents) required to answer the question. We consider retrieval successful
# if at least one of the retrieved chunks comes from one of these gold titles.
    
def compute_recall_at_k_supporting_titles(retrieved_chunks, supporting_facts):
    """
    Recall@k based on supporting document titles (HotpotQA).

    Returns:
    1 if at least one retrieved chunk has a title matching a gold supporting title
    0 otherwise
    """

    # Extract gold titles and remove duplicates using a set
    gold_titles = set(supporting_facts["title"])

    # Check if any retrieved chunk matches one of the gold titles
    for chunk in retrieved_chunks:
        retrieved_title = chunk.get("title", None)

        if retrieved_title in gold_titles:
            return 1

    return 0

In [18]:
# Test metrics on the current single example

em = compute_em(generated_answer, correct_answer)
f1 = compute_f1(generated_answer, correct_answer)
recall_k = compute_recall_at_k(retrieved_chunks, correct_answer)
recall_title = compute_recall_at_k_supporting_titles( retrieved_chunks, example["supporting_facts"])

print("Generated answer:", generated_answer)
print("Correct answer:", correct_answer)
print("EM:", em)
print("F1:", f1)
print("Recall@k:", recall_k)
print("Recall@k (support titles):", recall_title)

Generated answer: No.
Correct answer: yes
EM: 0
F1: 0.0
Recall@k: 0
Recall@k (support titles): 1


In [19]:
def run_single_example(example, chunk_size, top_k):
    """
    Run full RAG pipeline for one example.

    Returns a dictionary with:
    - prediction
    - correct answer
    - EM
    - F1
    - Recall@k
    - latency
    """

    question = example["question"]
    correct_answer = example["answer"]

    # Select correct index + metadata for this chunk size
    index = faiss_indexes[chunk_size]
    metadata = all_metadata[chunk_size]

    # --- Retrieval ---
    retrieved_chunks, _ = retrieve_top_k(
        question=question,
        index=index,
        metadata=metadata,
        k=top_k
    )

    # --- Context building ---
    context = build_retrieved_context(retrieved_chunks)

    # --- Prompt ---
    prompt = build_prompt(question, context)

    # --- Generation + latency ---
    start_time = time.time()

    prediction = generate_answer(
        prompt,
        tokenizer=llm_tokenizer,
        model=llm_model
    )

    latency = time.time() - start_time

    # --- Metrics ---
    em = compute_em(prediction, correct_answer)
    f1 = compute_f1(prediction, correct_answer)
    recall_k = compute_recall_at_k(retrieved_chunks, correct_answer)
    recall_title = compute_recall_at_k_supporting_titles( retrieved_chunks, example["supporting_facts"])

    return {
        "question": question,
        "prediction": prediction,
        "correct_answer": correct_answer,
        "EM": em,
        "F1": f1,
        "Recall@k": recall_k,
        "Recall@k_titles": recall_title,
        "latency": latency
    }

In [20]:
def run_test_evaluation(debug_sample, chunk_size, top_k):
    """
    Run evaluation on a small subset of questions.
    """

    results = []

    for i, example in enumerate(debug_sample):
        print(f"Running example {i+1}/{len(debug_sample)}")

        result = run_single_example(
            example=example,
            chunk_size=chunk_size,
            top_k=top_k
        )

        results.append(result)

    return results

In [21]:
results = run_test_evaluation(
    debug_sample=debug_sample,
    chunk_size=debug_chunk_size,
    top_k=debug_top_k
)


Running example 1/5
Running example 2/5
Running example 3/5
Running example 4/5
Running example 5/5


In [22]:
import numpy as np

def summarize_results(results):
    em = np.mean([r["EM"] for r in results])
    f1 = np.mean([r["F1"] for r in results])
    recall = np.mean([r["Recall@k"] for r in results])
    recall_title = np.mean([r["Recall@k_titles"] for r in results])

    latencies = [r["latency"] for r in results]
    p50 = np.percentile(latencies, 50)
    p95 = np.percentile(latencies, 95)

    return {
        "EM": em,
        "F1": f1,
        "Recall@k": recall,
        "Recall@k_titles": recall_title,
        "Latency_p50": p50,
        "Latency_p95": p95
    }
summary = summarize_results(results)

print("\n=== SUMMARY ===")
for k, v in summary.items():
    print(f"{k}: {v:.4f}")


=== SUMMARY ===
EM: 0.0000
F1: 0.1808
Recall@k: 0.4000
Recall@k_titles: 0.8000
Latency_p50: 15.7610
Latency_p95: 19.7774


In [25]:
# Inspect each example in detail

for i, r in enumerate(results):
    print(f"\n===== Example {i+1} =====")
    print("Question:", r["question"])
    print("Prediction:", r["prediction"])
    print("Ground truth:", r["correct_answer"])
    print("EM:", r["EM"])
    print("F1:", r["F1"])
    print("Recall@k:", r["Recall@k"])
    print("Latency:", f"{r['latency']:.2f}s")


===== Example 1 =====
Question: Were Scott Derrickson and Ed Wood of the same nationality?
Prediction: No.
Ground truth: yes
EM: 0
F1: 0.0
Recall@k: 0
Latency: 11.45s

===== Example 2 =====
Question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Prediction: Hillary Clinton
Ground truth: Chief of Protocol
EM: 0
F1: 0.0
Recall@k: 0
Latency: 15.76s

===== Example 3 =====
Question: What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?
Prediction: Animorphs, Northern Lights, Isaac Asimov's Robots and Aliens, Roswell High, The Rowan.
Ground truth: Animorphs
EM: 0
F1: 0.15384615384615385
Recall@k: 1
Latency: 20.26s

===== Example 4 =====
Question: Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?
Prediction: Yes, they are.
Ground truth: no
EM: 0
F1: 0.0
Recall@k: 1
Latency: 14.38s

===== Example 5 =====
Questi